# Telco customer churn: baseline model

Goal: predict `Churn`. Split into training and test sets first, learn the preprocessing rules from the training set only, then train a model and read the results.

This first baseline is evaluated directly on the test set. That is why the later notebooks select the pipeline and the threshold on a separate validation set, and why the README states that the test partition is not a fully untouched holdout.

## 1. Features and target

`y` is the churn outcome; `x` holds the customer attributes the model may use. The customer ID is not a feature.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", df.shape)
display(df[["customerID", "Contract", "tenure", "InternetService", "Churn"]].head())

Shape: (7043, 21)


,customerID,Contract,tenure,InternetService,Churn
0,7590-VHVEG,Month-to-month,1,DSL,No
1,5575-GNVDE,One year,34,DSL,No
2,3668-QPYBK,Month-to-month,2,DSL,Yes
3,7795-CFOCW,One year,45,DSL,No
4,9237-HQITU,Month-to-month,2,Fiber optic,Yes


In [2]:
y = df["Churn"]
x = df.drop(columns=["Churn", "customerID"])

display(y.head())
display(x.head())
print(len(x), len(y))

0     No
1     No
2    Yes
3     No
4    Yes
Name: Churn, dtype: str

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


7043 7043


## 2. Stratified train/test split

20% goes to the test set; `stratify=y` keeps the churn rate similar on both sides. A fixed random seed makes the split reproducible.

In [3]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=2002,
    stratify=y,
)

In [4]:
print("Train:", y_train.value_counts().to_dict())
print("Test: ", y_test.value_counts().to_dict())

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Train: {'No': 4139, 'Yes': 1495}
Test:  {'No': 1035, 'Yes': 374}
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


## 3. Numeric and categorical columns

`TotalCharges` is converted to numeric; blanks that cannot be converted stay missing. `SeniorCitizen` is stored as 0/1 but is treated as a category.

In [5]:
x_train = x_train.copy()
x_test = x_test.copy()

print("Numeric columns:", x_train.select_dtypes(include="number").columns.tolist())
print("Non-numeric columns:", x_train.select_dtypes(exclude="number").columns.tolist())

# Type conversion learns nothing from the data; missing values are imputed later from the training set
x_train["TotalCharges"] = pd.to_numeric(x_train["TotalCharges"], errors="coerce")
x_test["TotalCharges"] = pd.to_numeric(x_test["TotalCharges"], errors="coerce")

print(x_train["TotalCharges"].dtype)
print("Missing in train:", x_train["TotalCharges"].isna().sum())
print("Missing in test:", x_test["TotalCharges"].isna().sum())

# Numeric columns are named explicitly; text columns and the 0/1 category are one-hot encoded
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["SeniorCitizen"] + x_train.select_dtypes(
    exclude="number"
).columns.tolist()

print(numeric_features)
print(categorical_features)

Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Non-numeric columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']
float64
Missing in train: 7
Missing in test: 4
['tenure', 'MonthlyCharges', 'TotalCharges']
['SeniorCitizen', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 4. Learning the preprocessing rules from the training set only

Numeric columns: missing values are filled with each column's training median. Categorical columns: the one-hot layout is fixed from the training set. Both sets of rules are then applied to the training and the test data.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# fit sees the training set only and stores each numeric column's median
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

numeric_pipeline.fit(x_train[numeric_features])
print(numeric_pipeline.named_steps["imputer"].statistics_)

# transform reuses those medians and does not modify x_train or x_test
train_numeric = numeric_pipeline.transform(x_train[numeric_features])
test_numeric = numeric_pipeline.transform(x_test[numeric_features])
print("Missing in test after imputation:", pd.isna(test_numeric).sum())

# The encoder fixes the output columns from the training set; the test set follows the same order
all_cat_encoder = OneHotEncoder(handle_unknown="ignore")
all_cat_encoder.fit(x_train[categorical_features])

train_cat = all_cat_encoder.transform(x_train[categorical_features])
test_cat = all_cat_encoder.transform(x_test[categorical_features])

print("Train:", train_cat.shape)
print("Test:", test_cat.shape)

[  28.     70.3  1378.75]
Missing in test after imputation: 0
Train: (5634, 43)
Test: (1409, 43)


## 5. Combining the processed features

Numeric and encoded categorical results are placed side by side for each customer. Training and test sets must end up with the same number of columns.

In [7]:
from scipy.sparse import hstack

# Numeric columns first, encoded categorical columns after, on both sides
X_train_ready = hstack([train_numeric, train_cat]).tocsr()
X_test_ready = hstack([test_numeric, test_cat]).tocsr()

print("Train:", X_train_ready.shape)
print("Test:", X_test_ready.shape)

Train: (5634, 46)
Test: (1409, 46)


## 6. Logistic-regression baseline

The model learns from `X_train_ready` and `y_train` only; the test set is used for prediction and compared with `y_test`.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=2000)
model.fit(X_train_ready, y_train)

y_pred = model.predict(X_test_ready)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.85      0.90      0.87      1035
         Yes       0.67      0.55      0.60       374

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



## 7. Confusion matrix: false alarms and misses

Rows are the actual outcome, columns the prediction, both ordered `No`, `Yes`. The top-right cell holds false alarms (false positives), the bottom-left cell missed churners (false negatives).

In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])

display(pd.DataFrame(
    cm,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
))

,Predicted No,Predicted Yes
Actual No,935,100
Actual Yes,169,205


## Result

For the churn class the baseline reaches precision 0.67, recall 0.55 and F1 0.60, with an overall accuracy of 0.81: 935 true negatives, 100 false alarms, 169 missed churners and 205 correctly flagged churners.

The default 0.5 threshold misses 45% of churners. The next notebook looks at the threshold on a validation set carved out of the training data, without using the test set again.